# BRCA active-inference example

This notebook is the cleaned paper workflow. The original exploratory notebook is preserved at `archive/BRCA_active_original.ipynb`. Shared numerical code lives in `utils.py`; shared plotting code lives in `plotting.py`.


In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

from utils import (
    HUMAN_N_COL,
    EFFECTIVE_N_COL,
    run_odds_ratio_monte_carlo,
    summarize_monte_carlo,
)
from plotting import (
    make_monte_carlo_variance_table,
    plot_coverage,
    plot_effective_sample_size,
    plot_effective_sample_size_multiplier,
    plot_finite_population_coverage,
    plot_intervals,
    plot_monte_carlo_variance,
    plot_monte_carlo_variance_components,
    save_legend,
    save_monte_carlo_variance_table,
    set_theme_bw,
)


## Configuration


In [ ]:
DATA_PATH = Path("../Data/BRCA/master_df_withpred.csv")
PLOTS_DIR = Path("plots")
RESULTS_DIR = Path("results")
PLOTS_DIR.mkdir(exist_ok=True)
RESULTS_DIR.mkdir(exist_ok=True)

SEED = 614
ALPHA = 0.10
FRACS_HUMAN = np.linspace(0.2, 0.8, 20)
NUM_TRIALS = 500
TAU = 0.10


## Load data


In [ ]:
master_df = pd.read_csv(DATA_PATH)
master_df["strat"] = master_df["race"] + "_" + master_df["is_tnbc"].map({True: "TNBC", False: "nonTNBC"})

train_df, test_df = train_test_split(
    master_df,
    test_size=0.80,
    stratify=master_df["strat"],
    random_state=0,
)

master_df.shape, train_df.shape, test_df.shape


## Prediction diagnostics


In [ ]:
diagnostics = []
for race, group in master_df.groupby("race"):
    diagnostics.append(
        {
            "race": race,
            "n": len(group),
            "tnbc_rate": group["is_tnbc"].mean(),
            "cnn_accuracy": group["is_correct"].mean(),
            "cnn_auc": roc_auc_score(group["is_tnbc"].astype(int), group["predicted_probability"]),
        }
    )

diagnostics.append(
    {
        "race": "overall",
        "n": len(master_df),
        "tnbc_rate": master_df["is_tnbc"].mean(),
        "cnn_accuracy": master_df["is_correct"].mean(),
        "cnn_auc": roc_auc_score(master_df["is_tnbc"].astype(int), master_df["predicted_probability"]),
    }
)

pd.DataFrame(diagnostics)


## Prepare odds-ratio estimand

Group 1 is `W`; group 0 is all non-`W` rows. The estimand is the odds ratio of TNBC prevalence for group 1 versus group 0.


In [ ]:
yhat = master_df["predicted_probability"].to_numpy(dtype=float)
y = master_df["is_tnbc"].to_numpy(dtype=float)
group1 = master_df["race"].to_numpy() == "W"

yhat1, yhat0 = yhat[group1], yhat[~group1]
y1, y0 = y[group1], y[~group1]
n0, n1 = len(y0), len(y1)
n = n0 + n1

mu0 = y0.mean()
mu1 = y1.mean()
true_odds_ratio = (mu1 / (1 - mu1)) / (mu0 / (1 - mu0))
true_variance = (
    1 / np.sum(y0 == 0)
    + 1 / np.sum(y0 == 1)
    + 1 / np.sum(y1 == 0)
    + 1 / np.sum(y1 == 1)
) * n

mu0_train = train_df.loc[train_df["race"] != "W", "is_tnbc"].mean()
mu1_train = train_df.loc[train_df["race"] == "W", "is_tnbc"].mean()

{
    "n_group0": n0,
    "n_group1": n1,
    "mu0": mu0,
    "mu1": mu1,
    "true_odds_ratio": true_odds_ratio,
    "true_variance": true_variance,
    "mu0_train": mu0_train,
    "mu1_train": mu1_train,
}


## Monte Carlo comparison

The result table records the point estimate, log point estimate, analytic variance estimate, interval endpoints, interval width, coverage, effective sample size, empirical Monte Carlo variance by method and budget, and the finite-population-calibrated interval used to evaluate coverage against the fixed empirical population.


In [ ]:
df = run_odds_ratio_monte_carlo(
    y0=y0,
    yhat0=yhat0,
    y1=y1,
    yhat1=yhat1,
    fracs_human=FRACS_HUMAN,
    alpha=ALPHA,
    num_trials=NUM_TRIALS,
    true_odds_ratio=true_odds_ratio,
    true_variance=true_variance,
    mu0_pilot=mu0_train,
    mu1_pilot=mu1_train,
    tau=TAU,
    seed=SEED,
    split_spline_budget_evenly=True,
    show_progress=True,
)

summary_df = summarize_monte_carlo(df)
mc_variance_table = make_monte_carlo_variance_table(df)

df.to_csv(RESULTS_DIR / "BRCA_results.csv", index=False)
summary_df.to_csv(RESULTS_DIR / "BRCA_monte_carlo_summary.csv", index=False)
mc_variance_table.to_csv(RESULTS_DIR / "BRCA_monte_carlo_variance_components.csv", index=False)

mc_variance_table.head(12)


## Plots


In [ ]:
set_theme_bw()
plot_effective_sample_size(
    df,
    path=PLOTS_DIR / "BRCA_effective_sample_size.pdf",
    n_total=n,
    error_bars="sd",
)
plot_effective_sample_size(
    df,
    path=PLOTS_DIR / "BRCA_effective_sample_size_no_error_bars.pdf",
    n_total=n,
    error_bars="none",
)
plot_effective_sample_size_multiplier(
    df,
    path=PLOTS_DIR / "BRCA_effective_sample_size_multiplier.pdf",
    n_total=n,
    error_bars="sd",
)
plot_effective_sample_size_multiplier(
    df,
    path=PLOTS_DIR / "BRCA_effective_sample_size_multiplier_no_error_bars.pdf",
    n_total=n,
    error_bars="none",
)
plot_coverage(
    df,
    alpha=ALPHA,
    path=PLOTS_DIR / "BRCA_coverage.pdf",
    n_total=n,
)
plot_finite_population_coverage(
    df,
    alpha=ALPHA,
    path=PLOTS_DIR / "BRCA_coverage_finite_population_calibrated.pdf",
    n_total=n,
)
plot_monte_carlo_variance(
    df,
    path=PLOTS_DIR / "BRCA_mc_log_variance.pdf",
    use_log=True,
    n_total=n,
)
plot_monte_carlo_variance_components(
    df,
    path=PLOTS_DIR / "BRCA_mc_variance_components.pdf",
    n_total=n,
)
save_monte_carlo_variance_table(
    df,
    path=PLOTS_DIR / "BRCA_mc_variance_components_table.png",
)
plot_intervals(
    df,
    true_value=true_odds_ratio,
    path=PLOTS_DIR / "BRCA_intervals.pdf",
    estimand_label=r"odds ratio $O_{W/B}$",
    n_index=-1,
    num_intervals=5,
    seed=SEED,
)
save_legend(PLOTS_DIR / "BRCA_legend.pdf")


## Stability summary


In [ ]:
mc_variance_table
